# Objectives

- Main goal: If a person eats 100g of each seaweed, what percentage of his/her daily AS or LSS does he/she get?
- Show ANSES nutritional reference values for specific nutrients
- These reference values determine the quantity (in grams, milligrams or micgrograms) of each nutrient it is recommended to consume daily
- The charts show what percentage of reference value, for each nutrient, 100g of each seaweed corresponds to
- This way we can see that 100g of Wakamé atlantique provides 44,1% of daily recommended iron intake

**INFO:** work started in this notebook and continued in algues_apports-nutritionnels_vs_quantites-journalieres-recommandees.ipynb

## **MAIN Source:**
- ANSES: https://www.anses.fr/fr/system/files/NUT2012SA0103Ra-1.pdf, p. 12


A few acronyms:
- **BNM** = Besoin Nutritionnel Moyen (BNM)
    - "besoin quotidien moyen au sein de la population"
- **RNP** = Référence Nutritionnelle pour la Population
    - "apport quotidien qui couvre le besoin de presque toute la population considérée"
- **AS** = l’Apport Satisfaisant
    - "apport quotidien moyen d’une population ou d’un sous-groupe
pour lequel le statut nutritionnel est jugé satisfaisant"
      - "L’AS est la référence nutritionnelle retenue :
        - quand le BNM et donc la RNP ne peuvent pas être estimés faute de données suffisantes
        - ou quand la valeur de RNP peut être estimée mais n’est pas jugée satisfaisante..."
- **IR** = Intervalles de Référence
    - "référence nutritionnelle spécifique aux macronutriments énergétiques, exprimée en pourcentage de l'apport énergétique total"
- **LSS** = Limites Supérieures de Sécurité
    - "apport journalier chronique maximal d'une vitamine ou d’un minéral considéré comme peu susceptible de présenter un risque d'effets indésirables sur la santé de toute la population"


# imports

In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


from lxml import etree # for importing other datRNPet that hRNP 'alim_nom_fr' + 'ALIM_INDEX_FR' columns

import unicodedata # for cleaning CEVA df before merging

In [24]:
pd.set_option('display.max_rows', None)

## cleaned combined (CEVA + ciqual base) df 

In [25]:
df_combined_cleaned = pd.read_csv('data/processed/df_ciqual_ceva_seaweeds_cleaned')

df_combined_cleaned.head()

,alim_code,alim_grp_code,alim_grp_nom_eng,alim_ssgrp_nom_eng,alim_nom_eng,alim_nom_fr,"Energy, Regulation EU No 1169/2011 (kJ/100g)","Energy, Regulation EU No 1169/2011 (kcal/100g)",Water (g/100g),Fibres (g/100g),...,Vitamin C (mg/100g),Vitamin D (µg/100g),Vitamin E (mg/100g),Vitamin K1 (µg/100g),Zinc (mg/100g),Iodine (µg/100g),approx_count,seaweed_type,seaweed_name_Eng,seaweed_name_French
0,10041.0,4.0,"meat, egg and fish","seafood, raw","Abalone or ormer or sea ear, raw","Ormeau, cru",422.0,99.6,74.6,0.0000,...,2.000000,0.0000,4.00,NaN,0.82,NaN,0,unknown,"Abalone or ormer or sea ear, raw","Ormeau, cru"
1,12112.0,5.0,milk and milk products,cheese and similar,"Abondance cheese, from cow's milk",Abondance,1630.0,393.0,37.2,0.0000,...,0.375000,0.1875,0.75,2.25,3.50,15.0,6,unknown,"Abondance cheese, from cow's milk",Abondance
2,13404.0,2.0,"fruits, vegetables, legumes and nuts",fruits,"Acerola, pulp, raw, sampled in the island of L...","Cerise acérola, pulpe, crue, prélevée à la Ma...",NaN,NaN,93.1,NaN,...,2850.000000,NaN,NaN,NaN,NaN,NaN,0,unknown,"Acerola, pulp, raw, sampled in the island of L...","Cerise acérola, pulpe, crue, prélevée à la Ma..."
3,11168.0,10.0,miscellaneous,sauces,"Aioli sauce (garlic and olive oil mayonnaise),...","Sauce aïoli, préemballée",NaN,NaN,NaN,0.4200,...,NaN,NaN,NaN,NaN,NaN,NaN,0,unknown,"Aioli sauce (garlic and olive oil mayonnaise),...","Sauce aïoli, préemballée"
4,26006.0,4.0,"meat, egg and fish","fish, raw","Alaska pollock, raw","Lieu ou colin d'Alaska, cru",300.0,70.8,81.2,0.1275,...,0.000001,1.1000,0.36,0.10,0.41,78.2,1,unknown,"Alaska pollock, raw","Lieu ou colin d'Alaska, cru"


## filter dataset to only contain reduced nutrients list

In [26]:
df_combined_clean = df_combined_cleaned.rename(columns={
    'Protein, crude, N x 6.25 (g/100g)': 'Protein (g/100g)'
})

In [27]:
# Trimmed down nutrients for public-friendly comparison
nutrients_list_step1 = [
    "Fibres (g/100g)",
    "Protein (g/100g)",
    "Calcium (mg/100g)",
    "Iodine (µg/100g)",
    "Fat (g/100g)",
    "FA saturated (g/100g)",
    "Iron (mg/100g)"
]

In [28]:
df = df_combined_clean[
    ['alim_code', 'alim_nom_eng', 'alim_nom_fr', 'alim_ssgrp_nom_eng', 'seaweed_type'] + nutrients_list_step1
]

df.head()

,alim_code,alim_nom_eng,alim_nom_fr,alim_ssgrp_nom_eng,seaweed_type,Fibres (g/100g),Protein (g/100g),Calcium (mg/100g),Iodine (µg/100g),Fat (g/100g),FA saturated (g/100g),Iron (mg/100g)
0,10041.0,"Abalone or ormer or sea ear, raw","Ormeau, cru","seafood, raw",unknown,0.0000,14.40,31.0,NaN,0.830,0.150,5.90
1,12112.0,"Abondance cheese, from cow's milk",Abondance,cheese and similar,unknown,0.0000,26.60,760.0,15.0,31.600,20.100,0.09
2,13404.0,"Acerola, pulp, raw, sampled in the island of L...","Cerise acérola, pulpe, crue, prélevée à la Ma...",fruits,unknown,NaN,0.56,NaN,NaN,0.097,NaN,NaN
3,11168.0,"Aioli sauce (garlic and olive oil mayonnaise),...","Sauce aïoli, préemballée",sauces,unknown,0.4200,1.13,NaN,NaN,41.000,NaN,NaN
4,26006.0,"Alaska pollock, raw","Lieu ou colin d'Alaska, cru","fish, raw",unknown,0.1275,16.30,12.8,78.2,0.610,0.015,0.40


## Extract essential data for RI percentage chart

In [29]:
df_RI_perc = df[['seaweed_type', 'alim_nom_fr', 'alim_ssgrp_nom_eng', 'Fibres (g/100g)', 'Protein (g/100g)', 'Calcium (mg/100g)', 'Iron (mg/100g)', 'Iodine (µg/100g)']]

df_RI_perc_filtered = df_RI_perc[df_RI_perc['alim_ssgrp_nom_eng'] == 'seaweed']

df_RI_perc_filtered = df_RI_perc_filtered.drop(columns='alim_ssgrp_nom_eng')

df_RI_perc_filtered.head()

,seaweed_type,alim_nom_fr,Fibres (g/100g),Protein (g/100g),Calcium (mg/100g),Iron (mg/100g),Iodine (µg/100g)
101,algue brune,"Wakamé atlantique (Alaria esculenta), séchée ...",42.9,12.2,771.0,44.6,362000.0
268,algue brune,"Fucus vésiculeux (Fucus vesiculosus), séché ou...",46.7,6.4,1256.0,13.1,40500.0
479,algue rouge,Lichen de mer ou pioca ou goémon rouge (Chond...,35.5,16.7,911.0,18.1,31300.0
943,algue rouge,"Dulse (Palmaria palmata), séchée ou déshydratée",29.2,16.9,577.0,29.3,229000.0
1258,algue rouge,"Gracilaire ou ogonori (Gracilaria verrucosa),...",25.4,17.8,770.0,36.0,355800.0


## Remove info about dried or dehydrated & remove "algue" from seaweed_type column

In [30]:
# Rename multiple columns
# df_RI_perc_filtered = df_RI_perc_filtered.rename(columns={
#     'Fibres (g/100g)': 'Fibres_g',
#     'Protein (g/100g)': 'Protein_g',
#     'Calcium (mg/100g)': 'Iron_mg',
#     'Iodine (µg/100g)': 'Iodine_µg'
# })

# remove info 'séchée' ou 'déshydraté' du nom de l'algue
df_RI_perc_filtered["alim_nom_fr"] = (df_RI_perc_filtered["alim_nom_fr"].str.split(",", n=1)).str[0]

df_RI_perc_filtered["seaweed_type"] = df_RI_perc_filtered["seaweed_type"].str.replace("algue ", "")

df_RI_perc_filtered.head()

,seaweed_type,alim_nom_fr,Fibres (g/100g),Protein (g/100g),Calcium (mg/100g),Iron (mg/100g),Iodine (µg/100g)
101,brune,Wakamé atlantique (Alaria esculenta),42.9,12.2,771.0,44.6,362000.0
268,brune,Fucus vésiculeux (Fucus vesiculosus),46.7,6.4,1256.0,13.1,40500.0
479,rouge,Lichen de mer ou pioca ou goémon rouge (Chond...,35.5,16.7,911.0,18.1,31300.0
943,rouge,Dulse (Palmaria palmata),29.2,16.9,577.0,29.3,229000.0
1258,rouge,Gracilaire ou ogonori (Gracilaria verrucosa),25.4,17.8,770.0,36.0,355800.0


## Rename columns for clarity

In [31]:
#Rename multiple columns
df_RI_perc_filtered = df_RI_perc_filtered.rename(columns={
    "seaweed_type": "Type d'algue",
    "alim_nom_fr": "Algue",
})

df_RI_perc_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19 entries, 101 to 3042
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Type d'algue       19 non-null     object 
 1   Algue              19 non-null     object 
 2   Fibres (g/100g)    18 non-null     float64
 3   Protein (g/100g)   19 non-null     float64
 4   Calcium (mg/100g)  19 non-null     float64
 5   Iron (mg/100g)     19 non-null     float64
 6   Iodine (µg/100g)   17 non-null     float64
dtypes: float64(5), object(2)
memory usage: 1.2+ KB


## ANSES nutritional reference values - AS/RNP & LSS for adult women

In [32]:
from numpy import nan

AS_dict_women = {
    "Fibres (g/100g)": 25,      # g/day,  Sources: https://www.anses.fr/system/files/NUT2012SA0103Ra-2.pdf, https://www.anses.fr/system/files/NUT-Ra-Fibres.pdf
    "Protein (g/100g)": 49.8,     # g/day pour une femme de 60 kg -->. ANC protein = 0.83 g/kg poids corporel, Source:
    "Calcium (mg/100g)": 950,   # mg/day, Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
    "Iron (mg/100g)": 16,       # mg/day (women, varies by population), Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
    "Iodine (µg/100g)": 150     # µg/day, Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
}

LSS_dict_women = {
    "Fibres (g/100g)": nan,      # g/day (tolerable upper level, if available)
    "Protein (g/100g)": nan,    # g/day
    "Calcium (mg/100g)": 2500,  # mg/day
    "Iron (mg/100g)": 40,       # mg/day, Source: l'avis de l'Efsa de 2024
    "Iodine (µg/100g)": 600     # µg/day
}


AS_dict_men = {
    "Fibres (g/100g)": 25,      # g/day, Source: https://www.anses.fr/system/files/NUT2012SA0103Ra-2.pdf, https://www.anses.fr/system/files/NUT-Ra-Fibres.pdf
    "Protein (g/100g)": 58.1,     # g/day pour un homme de 70 kg -->. ANC protein = 0.83 g/kg poids corporel, Source:https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
    "Calcium (mg/100g)": 950,   # mg/day, Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
    "Iron (mg/100g)": 11,       # mg/day (women, varies by population), Source: 
    "Iodine (µg/100g)": 150     # µg/day, Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
}

LSS_dict_men = {
    "Fibres (g/100g)": nan,      # g/day (tolerable upper level, if available)
    "Protein (g/100g)": nan,    # g/day
    "Calcium (mg/100g)": 2500,  # mg/day
    "Iron (mg/100g)": 40,       # mg/day, Source: l'avis de l'Efsa de 2024, https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
    "Iodine (µg/100g)": 600     # µg/day, Source: https://www.anses.fr/fr/content/les-references-nutritionnelles-en-vitamines-et-mineraux
}




## Step 1 - Calculate %AS/RNP and %LSS

In [33]:
df_women = df_RI_perc_filtered.copy()

for nutrient in AS_dict_women.keys():
    as_val_women = AS_dict_women[nutrient]
    lss_val_women = LSS_dict_women[nutrient]

    # Compute %AS and %LSS
    df_women[f"%AS/RNP_{nutrient}"] = df_women[nutrient] / as_val_women * 100
    df_women[f"%LSS_{nutrient}"] = df_women[nutrient] / lss_val_women * 100

# Round for readability
df_women = df_women.round(1)

df_women.head()


,Type d'algue,Algue,Fibres (g/100g),Protein (g/100g),Calcium (mg/100g),Iron (mg/100g),Iodine (µg/100g),%AS/RNP_Fibres (g/100g),%LSS_Fibres (g/100g),%AS/RNP_Protein (g/100g),%LSS_Protein (g/100g),%AS/RNP_Calcium (mg/100g),%LSS_Calcium (mg/100g),%AS/RNP_Iron (mg/100g),%LSS_Iron (mg/100g),%AS/RNP_Iodine (µg/100g),%LSS_Iodine (µg/100g)
101,brune,Wakamé atlantique (Alaria esculenta),42.9,12.2,771.0,44.6,362000.0,171.6,NaN,24.5,NaN,81.2,30.8,278.8,111.5,241333.3,60333.3
268,brune,Fucus vésiculeux (Fucus vesiculosus),46.7,6.4,1256.0,13.1,40500.0,186.8,NaN,12.9,NaN,132.2,50.2,81.9,32.8,27000.0,6750.0
479,rouge,Lichen de mer ou pioca ou goémon rouge (Chond...,35.5,16.7,911.0,18.1,31300.0,142.0,NaN,33.5,NaN,95.9,36.4,113.1,45.2,20866.7,5216.7
943,rouge,Dulse (Palmaria palmata),29.2,16.9,577.0,29.3,229000.0,116.8,NaN,33.9,NaN,60.7,23.1,183.1,73.2,152666.7,38166.7
1258,rouge,Gracilaire ou ogonori (Gracilaria verrucosa),25.4,17.8,770.0,36.0,355800.0,101.6,NaN,35.7,NaN,81.1,30.8,225.0,90.0,237200.0,59300.0


In [34]:
df_men = df_RI_perc_filtered.copy()

for nutrient in AS_dict_men.keys():
    as_val_men = AS_dict_men[nutrient]
    lss_val_men = LSS_dict_men[nutrient]

    # Compute %AS and %LSS
    df_men[f"%AS/RNP_{nutrient}"] = df_men[nutrient] / as_val_men * 100
    df_men[f"%LSS_{nutrient}"] = df_men[nutrient] / lss_val_men * 100

# Round for readability
df_men = df_men.round(1)

df_men.head()


,Type d'algue,Algue,Fibres (g/100g),Protein (g/100g),Calcium (mg/100g),Iron (mg/100g),Iodine (µg/100g),%AS/RNP_Fibres (g/100g),%LSS_Fibres (g/100g),%AS/RNP_Protein (g/100g),%LSS_Protein (g/100g),%AS/RNP_Calcium (mg/100g),%LSS_Calcium (mg/100g),%AS/RNP_Iron (mg/100g),%LSS_Iron (mg/100g),%AS/RNP_Iodine (µg/100g),%LSS_Iodine (µg/100g)
101,brune,Wakamé atlantique (Alaria esculenta),42.9,12.2,771.0,44.6,362000.0,171.6,NaN,21.0,NaN,81.2,30.8,405.5,111.5,241333.3,60333.3
268,brune,Fucus vésiculeux (Fucus vesiculosus),46.7,6.4,1256.0,13.1,40500.0,186.8,NaN,11.0,NaN,132.2,50.2,119.1,32.8,27000.0,6750.0
479,rouge,Lichen de mer ou pioca ou goémon rouge (Chond...,35.5,16.7,911.0,18.1,31300.0,142.0,NaN,28.7,NaN,95.9,36.4,164.5,45.2,20866.7,5216.7
943,rouge,Dulse (Palmaria palmata),29.2,16.9,577.0,29.3,229000.0,116.8,NaN,29.1,NaN,60.7,23.1,266.4,73.2,152666.7,38166.7
1258,rouge,Gracilaire ou ogonori (Gracilaria verrucosa),25.4,17.8,770.0,36.0,355800.0,101.6,NaN,30.6,NaN,81.1,30.8,327.3,90.0,237200.0,59300.0


### Reorder columns

In [35]:
# Nutrients of interest
nutrients = ["Fibres (g/100g)", "Protein (g/100g)", "Calcium (mg/100g)", "Iron (mg/100g)", "Iodine (µg/100g)"]

# Build desired column order
new_order = ["Type d'algue", "Algue"]  # keep identifiers first
for n in nutrients:
    new_order += [n, f"%AS/RNP_{n}", f"%LSS_{n}"]

# Reorder columns in dataframe
df_women = df_women[new_order]


# Remove Agar & Spiruline from list bc they are not part of approved edible seaweeds
df_women_15 = df_women[
    ~(
        df_women['Algue'].str.contains('Spir') | df_women['Algue'].str.contains('Agar')
        )
        ]


print(df_women.shape)
print(df_women_15.shape)

(19, 17)
(16, 17)


In [36]:
# Reorder columns in dataframe 
df_men = df_men[new_order]

# Remove Agar & Spiruline from list bc they are not part of approved edible seaweeds
df_men_15 = df_men[
    ~(
        df_men['Algue'].str.contains('Spir') | df_men['Algue'].str.contains('Agar')
        )
        ]


print(df_men.shape)
print(df_men_15.shape)

(19, 17)
(16, 17)


In [37]:
df_display_women = df_women_15.rename(columns={
    "Algue": "Macroalgue",
    "Fibres (g/100g)": "Fibres (g/100 g)*",
    "%AS/RNP_Fibres (g/100g)": "%AS/RNP Fibres",
    "%LSS_Fibres (g/100g)": "%LSS Fibres",
    "Protein (g/100g)": "Protéines (g/100 g)*",
    "%AS/RNP_Protein (g/100g)": "%AS/RNP Protéines",
    "%LSS_Protein (g/100g)": "%LSS Protéines",
    "Calcium (mg/100g)": "Calcium (mg/100 g)*",
    "%AS/RNP_Calcium (mg/100g)": "%AS/RNP Calcium",
    "%LSS_Calcium (mg/100g)": "%LSS Calcium",
    "Iron (mg/100g)": "Fer (mg/100 g)*",
    "%AS/RNP_Iron (mg/100g)": "%AS/RNP Fer",
    "%LSS_Iron (mg/100g)": "%LSS Fer",
    "Iodine (µg/100g)": "Iode (µg/100 g)*",
    "%AS/RNP_Iodine (µg/100g)": "%AS/RNP Iode",
    "%LSS_Iodine (µg/100g)": "%LSS Iode"
})



df_display_women = df_display_women.sort_values("Type d'algue")

df_display_women

,Type d'algue,Macroalgue,Fibres (g/100 g)*,%AS/RNP Fibres,%LSS Fibres,Protéines (g/100 g)*,%AS/RNP Protéines,%LSS Protéines,Calcium (mg/100 g)*,%AS/RNP Calcium,%LSS Calcium,Fer (mg/100 g)*,%AS/RNP Fer,%LSS Fer,Iode (µg/100 g)*,%AS/RNP Iode,%LSS Iode
101,brune,Wakamé atlantique (Alaria esculenta),42.9,171.6,NaN,12.2,24.5,NaN,771.0,81.2,30.8,44.6,278.8,111.5,362000.0,241333.3,60333.3
268,brune,Fucus vésiculeux (Fucus vesiculosus),46.7,186.8,NaN,6.4,12.9,NaN,1256.0,132.2,50.2,13.1,81.9,32.8,40500.0,27000.0,6750.0
1431,brune,Kombu ou kombu japonais (Laminaria japonica),34.1,136.4,NaN,7.9,15.9,NaN,811.0,85.4,32.4,13.0,81.2,32.5,232800.0,155200.0,38800.0
1849,brune,Ascophylle noueux ou goémon noir (Ascophyllum...,43.6,174.4,NaN,7.4,14.9,NaN,1601.0,168.5,64.0,19.3,120.6,48.3,68600.0,45733.3,11433.3
2516,brune,Kombu royal (Saccharina latissima),30.2,120.8,NaN,9.9,19.9,NaN,838.0,88.2,33.5,24.1,150.6,60.2,410000.0,273333.3,68333.3
2520,brune,Haricot de mer (Himanthalia elongata),30.8,123.2,NaN,9.9,19.9,NaN,803.0,84.5,32.1,2.1,13.1,5.3,9000.0,6000.0,1500.0
2817,brune,Kombu breton (Laminaria digitata),37.3,149.2,NaN,8.9,17.9,NaN,918.0,96.6,36.7,10.5,65.6,26.2,458800.0,305866.7,76466.7
2892,brune,Fucus vésiculeux (Fucus serratus),30.8,123.2,NaN,11.5,23.1,NaN,1187.0,124.9,47.5,14.5,90.6,36.2,40500.0,27000.0,6750.0
2893,brune,Fucus vésiculeux (Fucus serratus ou Fucus ves...,44.6,178.4,NaN,7.4,14.9,NaN,1170.0,123.2,46.8,14.7,91.9,36.8,40000.0,26666.7,6666.7
3042,brune,Wakamé (Undaria pinnatifida),36.9,147.6,NaN,14.0,28.1,NaN,963.0,101.4,38.5,15.5,96.9,38.8,18000.0,12000.0,3000.0


In [38]:
df_display_men = df_men_15.rename(columns={
    "Algue": "Macroalgue",
    "Fibres (g/100g)": "Fibres (g/100 g)*",
    "%AS/RNP_Fibres (g/100g)": "%AS/RNP Fibres",
    "%LSS_Fibres (g/100g)": "%LSS Fibres",
    "Protein (g/100g)": "Protéines (g/100 g)*",
    "%AS/RNP_Protein (g/100g)": "%AS/RNP Protéines",
    "%LSS_Protein (g/100g)": "%LSS Protéines",
    "Calcium (mg/100g)": "Calcium (mg/100 g)*",
    "%AS/RNP_Calcium (mg/100g)": "%AS/RNP Calcium",
    "%LSS_Calcium (mg/100g)": "%LSS Calcium",
    "Iron (mg/100g)": "Fer (mg/100 g)*",
    "%AS/RNP_Iron (mg/100g)": "%AS/RNP Fer",
    "%LSS_Iron (mg/100g)": "%LSS Fer",
    "Iodine (µg/100g)": "Iode (µg/100 g)*",
    "%AS/RNP_Iodine (µg/100g)": "%AS/RNP Iode",
    "%LSS_Iodine (µg/100g)": "%LSS Iode"
})



df_display_men = df_display_men.sort_values("Type d'algue")

df_display_men

,Type d'algue,Macroalgue,Fibres (g/100 g)*,%AS/RNP Fibres,%LSS Fibres,Protéines (g/100 g)*,%AS/RNP Protéines,%LSS Protéines,Calcium (mg/100 g)*,%AS/RNP Calcium,%LSS Calcium,Fer (mg/100 g)*,%AS/RNP Fer,%LSS Fer,Iode (µg/100 g)*,%AS/RNP Iode,%LSS Iode
101,brune,Wakamé atlantique (Alaria esculenta),42.9,171.6,NaN,12.2,21.0,NaN,771.0,81.2,30.8,44.6,405.5,111.5,362000.0,241333.3,60333.3
268,brune,Fucus vésiculeux (Fucus vesiculosus),46.7,186.8,NaN,6.4,11.0,NaN,1256.0,132.2,50.2,13.1,119.1,32.8,40500.0,27000.0,6750.0
1431,brune,Kombu ou kombu japonais (Laminaria japonica),34.1,136.4,NaN,7.9,13.6,NaN,811.0,85.4,32.4,13.0,118.2,32.5,232800.0,155200.0,38800.0
1849,brune,Ascophylle noueux ou goémon noir (Ascophyllum...,43.6,174.4,NaN,7.4,12.7,NaN,1601.0,168.5,64.0,19.3,175.5,48.3,68600.0,45733.3,11433.3
2516,brune,Kombu royal (Saccharina latissima),30.2,120.8,NaN,9.9,17.0,NaN,838.0,88.2,33.5,24.1,219.1,60.2,410000.0,273333.3,68333.3
2520,brune,Haricot de mer (Himanthalia elongata),30.8,123.2,NaN,9.9,17.0,NaN,803.0,84.5,32.1,2.1,19.1,5.3,9000.0,6000.0,1500.0
2817,brune,Kombu breton (Laminaria digitata),37.3,149.2,NaN,8.9,15.3,NaN,918.0,96.6,36.7,10.5,95.5,26.2,458800.0,305866.7,76466.7
2892,brune,Fucus vésiculeux (Fucus serratus),30.8,123.2,NaN,11.5,19.8,NaN,1187.0,124.9,47.5,14.5,131.8,36.2,40500.0,27000.0,6750.0
2893,brune,Fucus vésiculeux (Fucus serratus ou Fucus ves...,44.6,178.4,NaN,7.4,12.8,NaN,1170.0,123.2,46.8,14.7,133.6,36.8,40000.0,26666.7,6666.7
3042,brune,Wakamé (Undaria pinnatifida),36.9,147.6,NaN,14.0,24.1,NaN,963.0,101.4,38.5,15.5,140.9,38.8,18000.0,12000.0,3000.0


In [39]:
# FINAL PROPER WORKING CODE


# Convert floats to percentages as strings with "%"
for col in df_display_women.columns:
    if "%AS/RNP" in col or "%LSS" in col:
        df_display_women[col] = df_display_women[col].apply(
            lambda x: f"{int(round(x))}%" if pd.notna(x) else ""
        )

# Create MultiIndex for columns
col_mapping = {
    "Fibres (g/100 g)*": ("Fibres", "g"),
    "%AS/RNP Fibres": ("Fibres", "%RNP"),
    "%LSS Fibres": ("Fibres", "%LSS"),
    
    "Protéines (g/100 g)*": ("Protéines", "g"),
    "%AS/RNP Protéines": ("Protéines", "%RNP"),
    "%LSS Protéines": ("Protéines", "%LSS"),
    
    "Calcium (mg/100 g)*": ("Calcium", "mg"),
    "%AS/RNP Calcium": ("Calcium", "%RNP"),
    "%LSS Calcium": ("Calcium", "%LSS"),
    
    "Fer (mg/100 g)*": ("Fer", "mg"),
    "%AS/RNP Fer": ("Fer", "%RNP"),
    "%LSS Fer": ("Fer", "%LSS"),
    
    "Iode (µg/100 g)*": ("Iode", "µg"),
    "%AS/RNP Iode": ("Iode", "%RNP"),
    "%LSS Iode": ("Iode", "%LSS"),

    "Type d'algue": ("", "Type"),
    "Macroalgue": ("", "Macroalgue")
}

df_display_women.columns = pd.MultiIndex.from_tuples([col_mapping[c] for c in df_display_women.columns])

# Set Macroalgue as index
df_display_women = df_display_women.set_index(("", "Macroalgue"))

# Display
df_display_women.head()


Fibres             \
                                                     Type      g  %RNP %LSS   
(, Macroalgue)                                                                
 Wakamé atlantique (Alaria esculenta)               brune   42.9  172%        
Fucus vésiculeux (Fucus vesiculosus)                brune   46.7  187%        
 Kombu ou kombu japonais (Laminaria japonica)       brune   34.1  136%        
 Ascophylle noueux ou goémon noir (Ascophyllum ...  brune   43.6  174%        
 Kombu royal (Saccharina latissima)                 brune   30.2  121%        

                                                   Protéines            \
                                                           g %RNP %LSS   
(, Macroalgue)                                                           
 Wakamé atlantique (Alaria esculenta)                   12.2  24%        
Fucus vésiculeux (Fucus vesiculosus)                     6.4  13%        
 Kombu ou kombu japonais (Laminaria japonica)            7.9  16%        
 Ascophylle noueux ou goémon noir (Ascophyllum ...       7.4  15%        
 Kombu royal (Saccharina latissima)                      9.9  20%        

                                                   Calcium              Fer  \
                                                        mg  %RNP %LSS    mg   
(, Macroalgue)                                                                
 Wakamé atlantique (Alaria esculenta)                771.0   81%  31%  44.6   
Fucus vésiculeux (Fucus vesiculosus)                1256.0  132%  50%  13.1   
 Kombu ou kombu japonais (Laminaria japonica)        811.0   85%  32%  13.0   
 Ascophylle noueux ou goémon noir (Ascophyllum ...  1601.0  168%  64%  19.3   
 Kombu royal (Saccharina latissima)                  838.0   88%  34%  24.1   

                                                                    Iode  \
                                                    %RNP  %LSS        µg   
(, Macroalgue)                                                             
 Wakamé atlantique (Alaria esculenta)               279%  112%  362000.0   
Fucus vésiculeux (Fucus vesiculosus)                 82%   33%   40500.0   
 Kombu ou kombu japonais (Laminaria japonica)        81%   32%  232800.0   
 Ascophylle noueux ou goémon noir (Ascophyllum ...  121%   48%   68600.0   
 Kombu royal (Saccharina latissima)                 151%   60%  410000.0   

                                                                     
                                                       %RNP    %LSS  
(, Macroalgue)                                                       
 Wakamé atlantique (Alaria esculenta)               241333%  60333%  
Fucus vésiculeux (Fucus vesiculosus)                 27000%   6750%  
 Kombu ou kombu japonais (Laminaria japonica)       155200%  38800%  
 Ascophylle noueux ou goémon noir (Ascophyllum ...   45733%  11433%  
 Kombu royal (Saccharina latissima)                 273333%  68333%

In [40]:
# FINAL PROPER WORKING CODE

# Convert floats to percentages as strings with "%"
for col in df_display_men.columns:
    if "%AS/RNP" in col or "%LSS" in col:
        df_display_men[col] = df_display_men[col].apply(
            lambda x: f"{int(round(x))}%" if pd.notna(x) else ""
        )

# Create MultiIndex for columns
col_mapping = {
    "Fibres (g/100 g)*": ("Fibres", "g"),
    "%AS/RNP Fibres": ("Fibres", "%RNP"),
    "%LSS Fibres": ("Fibres", "%LSS"),
    
    "Protéines (g/100 g)*": ("Protéines", "g"),
    "%AS/RNP Protéines": ("Protéines", "%RNP"),
    "%LSS Protéines": ("Protéines", "%LSS"),
    
    "Calcium (mg/100 g)*": ("Calcium", "mg"),
    "%AS/RNP Calcium": ("Calcium", "%RNP"),
    "%LSS Calcium": ("Calcium", "%LSS"),
    
    "Fer (mg/100 g)*": ("Fer", "mg"),
    "%AS/RNP Fer": ("Fer", "%RNP"),
    "%LSS Fer": ("Fer", "%LSS"),
    
    "Iode (µg/100 g)*": ("Iode", "µg"),
    "%AS/RNP Iode": ("Iode", "%RNP"),
    "%LSS Iode": ("Iode", "%LSS"),

    "Type d'algue": ("", "Type"),
    "Macroalgue": ("", "Macroalgue")
}

df_display_men.columns = pd.MultiIndex.from_tuples([col_mapping[c] for c in df_display_men.columns])

# Set Macroalgue as index
df_display_men = df_display_men.set_index(("", "Macroalgue"))

# Display
df_display_men.head()


Fibres             \
                                                     Type      g  %RNP %LSS   
(, Macroalgue)                                                                
 Wakamé atlantique (Alaria esculenta)               brune   42.9  172%        
Fucus vésiculeux (Fucus vesiculosus)                brune   46.7  187%        
 Kombu ou kombu japonais (Laminaria japonica)       brune   34.1  136%        
 Ascophylle noueux ou goémon noir (Ascophyllum ...  brune   43.6  174%        
 Kombu royal (Saccharina latissima)                 brune   30.2  121%        

                                                   Protéines            \
                                                           g %RNP %LSS   
(, Macroalgue)                                                           
 Wakamé atlantique (Alaria esculenta)                   12.2  21%        
Fucus vésiculeux (Fucus vesiculosus)                     6.4  11%        
 Kombu ou kombu japonais (Laminaria japonica)            7.9  14%        
 Ascophylle noueux ou goémon noir (Ascophyllum ...       7.4  13%        
 Kombu royal (Saccharina latissima)                      9.9  17%        

                                                   Calcium              Fer  \
                                                        mg  %RNP %LSS    mg   
(, Macroalgue)                                                                
 Wakamé atlantique (Alaria esculenta)                771.0   81%  31%  44.6   
Fucus vésiculeux (Fucus vesiculosus)                1256.0  132%  50%  13.1   
 Kombu ou kombu japonais (Laminaria japonica)        811.0   85%  32%  13.0   
 Ascophylle noueux ou goémon noir (Ascophyllum ...  1601.0  168%  64%  19.3   
 Kombu royal (Saccharina latissima)                  838.0   88%  34%  24.1   

                                                                    Iode  \
                                                    %RNP  %LSS        µg   
(, Macroalgue)                                                             
 Wakamé atlantique (Alaria esculenta)               406%  112%  362000.0   
Fucus vésiculeux (Fucus vesiculosus)                119%   33%   40500.0   
 Kombu ou kombu japonais (Laminaria japonica)       118%   32%  232800.0   
 Ascophylle noueux ou goémon noir (Ascophyllum ...  176%   48%   68600.0   
 Kombu royal (Saccharina latissima)                 219%   60%  410000.0   

                                                                     
                                                       %RNP    %LSS  
(, Macroalgue)                                                       
 Wakamé atlantique (Alaria esculenta)               241333%  60333%  
Fucus vésiculeux (Fucus vesiculosus)                 27000%   6750%  
 Kombu ou kombu japonais (Laminaria japonica)       155200%  38800%  
 Ascophylle noueux ou goémon noir (Ascophyllum ...   45733%  11433%  
 Kombu royal (Saccharina latissima)                 273333%  68333%

In [41]:
df_display_women.to_excel("charts_references_nutritionnels/seaweed_nutrients_RV_women_Multi_index_old.xlsx", engine="openpyxl", index=True)
df_display_men.to_excel("charts_references_nutritionnels/seaweed_nutrients_RV_men_Multi_index_olf.xlsx", engine="openpyxl", index=True)
